# Validación de la capa plata

Transformación de las fuentes locales de bronce. Conserva originales y resultados privados.

Se mantienen las decisiones del equipo del 13-nov-2026: lotes fuera del ranking y GEM como cruce por cercanía con suma de empates. Los pesos pertenecen a oro. No certifica seguridad estructural.

In [ ]:
from pathlib import Path
import runpy, json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
raiz = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'notebooks/00_silver.py').exists())
modulo = runpy.run_path(str(raiz / 'scripts/correr_plata.py'))
tablas, reporte, destino = modulo['ejecutar_plata_local']()
print('Tablas privadas en:', destino)
print(json.dumps(reporte, ensure_ascii=False, indent=2))

## Tablas materializadas y puertas de calidad

Los controles fallidos detienen la escritura. Las tablas Delta se publican individualmente; la ejecución local produce Parquet y no implica una publicación en Databricks.

In [ ]:
display(pd.DataFrame([{'tabla': k, 'filas': len(v), 'columnas': len(v.columns)} for k,v in tablas.items()]))
display(tablas['calidad'])
assert tablas['calidad'].cumple.all()

## Conservación del inventario y pisos

NPISOS=0 conserva su significado catastral. Los pisos desconocidos se distinguen de los lotes; las estimaciones quedan separadas de los valores observados.

In [ ]:
p = tablas['predios']
resumen = p.groupby('estado_predial').agg(predios=('id','size'), pisos_observados=('pisos_observados','count'), pisos_estimados=('pisos_estimados','count'), con_senal_danio=('discrepancia_lote_satelite','sum'))
display(resumen)
display(p.groupby('pisos_fuente').size().rename('predios').reset_index())
assert len(p) == reporte['filas_bronze']
assert p.loc[p.sin_construccion_registrada, ['pisos','pisos_estimados','periodo_s']].isna().all().all()
ax = resumen.predios.plot.barh(color='#27678b', figsize=(9,3))
ax.bar_label(ax.containers[0], padding=4)
ax.set_xlim(0, resumen.predios.max()*1.13)
plt.title('Inventario conservado por estado predial'); plt.xlabel('Predios'); plt.ylabel(''); plt.tight_layout(); plt.show()

## Correspondencia GEM: política aceptada y límites

Se suman ocupantes/valor entre candidatos empatados por predio. Los atributos corresponden al candidato con más ocupantes, con desempate por ID. Un registro GEM puede participar en varios predios: estos ocupantes no son una población única censada y no deben sumarse como tal. No se acredita identidad catastral.

In [ ]:
gem = tablas['cruces_gem']
display(pd.Series({'predios_sin_candidato': int(p.n_registros_gem.eq(0).sum()), 'predios_con_empate': int(p.gem_ambiguo.sum()), 'predios_con_gem_reutilizado': int(p.gem_reutilizado.sum()), 'id_gem_en_varios_predios': int(gem.groupby('id_gem').id_predio.nunique().gt(1).sum()), 'relaciones_conservadas': len(gem)}, name='valor').to_frame())
p.n_registros_gem.clip(upper=8).value_counts().sort_index().plot.bar(color='#b87635', figsize=(8,3))
plt.title('Candidatos GEM a distancia mínima (8 agrupa 8 o más)'); plt.xlabel('Candidatos'); plt.ylabel('Predios'); plt.tight_layout(); plt.show()

## Territorio, salud y cobertura

Las distancias se calculan desde el contorno a la geometría objetivo en EPSG:9377. Son euclidianas, no tiempos de viaje. Salud mapeada no implica capacidad operativa; colegios no se declaran albergues. Una coincidencia territorial múltiple queda sin asignación automática.

In [ ]:
display(p[['n_coincidencias_comuna','n_coincidencias_barrio']].apply(pd.Series.value_counts).fillna(0))
display(tablas['servicios'].groupby(['fuente','tipo','urgencias']).size().rename('sitios').reset_index())
display(p[['dist_salud_m','dist_salud_urgencias_m','dist_bomberos_m','dist_via_principal_m','dist_microzona_m','dist_sitio_m','dist_shakemap_m']].describe(percentiles=[.5,.95]).T)
p[['dist_salud_m','dist_salud_urgencias_m']].rename(columns={'dist_salud_m':'Salud mapeada','dist_salud_urgencias_m':'Urgencias declaradas'}).plot.hist(bins=40, alpha=.55, figsize=(9,3))
plt.title('Proximidad a salud y urgencias declaradas'); plt.xlabel('Metros desde el contorno'); plt.ylabel('Predios'); plt.tight_layout(); plt.show()

## Sacudida, sitio y observaciones

ShakeMap está separado de las funciones GEM. PGA/PSA se convierten de %g a g. La asignación a grillas se limita a su envolvente convexa, conservando distancia y empates; esto no equivale a una delimitación geológica. No se introduce amplificación=1 cuando falta una función. Las observaciones son insumos de ShakeMap, no validación independiente.

In [ ]:
display(tablas['observaciones'].groupby('tipo_observacion').size().rename('observaciones').reset_index())
display(tablas['shakemap'][['pga_pct_g','pga_g','mmi']].describe().T)
display(p.groupby('estado_predial')[['periodo_s','amplif_suelo']].count())
display(pd.Series({'eventos_en_origen': len(tablas['eventos_origen']), 'eventos_unicos': len(tablas['eventos']), 'eventos_en_ambos_catalogos': int(tablas['eventos'].n_catalogos.eq(2).sum())}, name='valor').to_frame())

## Trazabilidad y comprobación del ranking existente

Se conservan huellas de archivos y del contenido procesado. Cuando el extracto no conserva un corte de origen, se declara desconocido. La comprobación siguiente reutiliza el índice existente sin cambiar sus pesos: verifica que los lotes continúan fuera del ranking.

In [ ]:
display(tablas['fuentes'][['fuente','filas_bronce','sha256_archivo','corte_estado']])
api = modulo['cargar_api']()
gold_prueba = api['calcular_indice'](p)
assert not gold_prueba.loc[gold_prueba.sin_construccion_registrada, 'en_ranking'].any()
assert not gold_prueba.loc[gold_prueba.sin_dato_satelital, 'en_ranking'].any()
display(gold_prueba.prioridad.value_counts().rename('predios').to_frame())
print(f"VALIDACIÓN OK: {len(tablas)} tablas, {len(p)} predios, {len(tablas['calidad'])} controles, fuentes intactas={reporte['fuentes_sin_modificacion']}")